# Лабораторная работа №5: CLIP: zero-shot-классификация и мультимодальный поиск

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.


## Цели обучения

После работы студент должен уметь:

- Сравнить prompt templates в zero-shot classification.
- Построить text-to-image retrieval.
- Дообучить projection head и оценить Recall@K.


## Теоретическая часть

CLIP обучает image encoder и text encoder в общем нормированном пространстве. Zero-shot logits пропорциональны cosine similarity между изображением и текстовыми шаблонами классов.

![CLIP](https://raw.githubusercontent.com/openai/CLIP/main/CLIP.png)


### Математическая постановка

$$s(x,t)=\frac{f_I(x)^\top f_T(t)}{\|f_I(x)\|\|f_T(t)\|},\qquad p(c\mid x)=\operatorname{softmax}(s(x,t_c)/\tau).$$
Для retrieval используются Recall@K и median rank.


### Материалы

- [OpenCLIP](https://github.com/mlfoundations/open_clip)
- [CLIP paper](https://arxiv.org/abs/2103.00020)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практическое задание

1. Сравнить prompt templates в zero-shot classification.
2. Построить text-to-image retrieval.
3. Дообучить projection head и оценить Recall@K.

**Обязательные артефакты:** код, конфигурация, метрики, минимум одна контролируемая ablation, примеры ошибок и краткие выводы.


### Профиль вычислений

- **Основной режим:** OpenCLIP ViT-B-32 frozen/малый projection head.
- **Ограничение данных:** CIFAR-10 или до 5 000 пар.
- **Fallback:** CPU на 500 объектах.
- Все длительные этапы должны сохранять checkpoint и продолжаться после перезапуска. Оценка не зависит от размера модели: важны корректность эксперимента и выводы.


**Оценивание.** Десять обязательных предметных этапов по 1 баллу: основная часть — 10 баллов. Творческий бонус — отдельно 1 балл, не заменяет обязательные этапы. Полный режим оценивается по реальным данным и обучению; `FAST_DEV_RUN=True` служит только smoke-проверкой, его показатели не являются отчётными.


## Реализация


In [46]:
# Базовое воспроизводимое окружение
from pathlib import Path
import json, os, random, time
import numpy as np

SEED = 42
FAST_DEV_RUN = False  # True — только проверка связности на уменьшенной выборке; не отчётный результат
ARTIFACTS = Path(os.getenv("CV_ARTIFACTS", "artifacts"))
ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch = None
    DEVICE = "cpu"
# Датасеты курса лежат в общем каталоге <корень курса>/data:
# notebook выполняется из labs/lab_XX_*/, поэтому корень курса — cwd.parent.parent.
# Для локальных запусков вне labs/ также поднимаемся, пока не найдём каталог data с CIFAR.
_CANDIDATES = [Path.cwd().parent.parent / "data", Path.cwd().parent / "data", Path.cwd() / "data"]
DATA_ROOT = Path(os.getenv("CV_COURSE_DATA") or next((c for c in _CANDIDATES if (c / "cifar-10-batches-py").exists()), _CANDIDATES[0]))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print({"device": DEVICE, "fast_dev_run": FAST_DEV_RUN, "data_root": str(DATA_ROOT)})


{'device': 'cpu', 'fast_dev_run': False, 'data_root': 'd:\\ML_SOTA\\data'}


In [47]:
def check(condition, message):
    """Мини-проверка в стиле YAAM: молчит, если всё хорошо; печатает, если найдена проблема."""
    if condition:
        print(f"✔ {message}")
    else:
        print(f"✘ ВНИМАНИЕ: {message}")

### Данные и предобученный CLIP

*Вес: 1 балл.*

**Постановка.** Индекс поиска и обучающие пары должны состоять из разных изображений.

Выполните следующие действия:
1. Загрузите CIFAR-10 train/test и OpenCLIP ViT-B-32.
2. Определите список классов и независимый test-загрузчик.

**Результат.** Объекты model, train_pairs, ds, classes, loader.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [48]:
# Реализуйте это задание по постановке выше.
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
import open_clip

MODEL_NAME = "ViT-B-32"
PRETRAINED = "openai"

model, _, preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME, pretrained=PRETRAINED
)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model = model.to(DEVICE).eval()
for p in model.parameters():
    p.requires_grad_(False)

classes = ["airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck"]

full_train = CIFAR10(root=DATA_ROOT, train=True,  download=False, transform=preprocess)
full_test  = CIFAR10(root=DATA_ROOT, train=False, download=False, transform=preprocess)

N_TRAIN = 100 if FAST_DEV_RUN else len(full_train)
N_TEST  = 100 if FAST_DEV_RUN else len(full_test)

rng = np.random.default_rng(SEED)
train_indices = rng.choice(len(full_train), size=N_TRAIN, replace=False).tolist()
test_indices  = rng.choice(len(full_test),  size=N_TEST,  replace=False).tolist()

# train_pairs — обучающие пары, ds — индекс поиска (test), loader — независимый test-загрузчик
train_pairs = Subset(full_train, train_indices)
ds          = Subset(full_test,  test_indices)
loader      = DataLoader(ds, batch_size=32, shuffle=False, num_workers=0)

check(len(train_pairs) > 0 and len(ds) > 0,
      f"CLIP {MODEL_NAME}/{PRETRAINED}: train={len(train_pairs)}, test={len(ds)}")
check(set(train_indices).isdisjoint(set(test_indices)),
      "Индексы train и test не пересекаются")

✔ CLIP ViT-B-32/openai: train=50000, test=10000
✘ ВНИМАНИЕ: Индексы train и test не пересекаются


### Текстовые прототипы классов

*Вес: 1 балл.*

**Постановка.** Нормализация эмбеддингов превращает скалярное произведение в cosine similarity.

Выполните следующие действия:
1. Создайте несколько шаблонов класса.
2. Закодируйте и нормируйте текстовые прототипы.

**Результат.** templates и функция text_prototypes.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [49]:
# Реализуйте это задание по постановке выше.
templates = [
    "a photo of a {}.",
    "a blurry photo of a {}.",
    "a photo of a small {}.",
    "a photo of a {} on a plain background.",
    "a close-up photo of a {}.",
]

@torch.no_grad()
def text_prototypes(model, tokenizer, classes, templates, device=DEVICE):
    """Средний по шаблонам нормированный прототип класса."""
    protos = []
    for cls in classes:
        texts = [t.format(cls) for t in templates]
        tokens = tokenizer(texts).to(device)
        feats = model.encode_text(tokens).float()
        feats = F.normalize(feats, dim=-1)
        proto = F.normalize(feats.mean(dim=0), dim=-1)
        protos.append(proto)
    return torch.stack(protos, dim=0)

text_feats = text_prototypes(model, tokenizer, classes, templates)

check(text_feats.shape[0] == len(classes),
      f"text_feats: shape={tuple(text_feats.shape)}, "
      f"средняя норма={text_feats.norm(dim=-1).mean():.3f}")

✔ text_feats: shape=(10, 512), средняя норма=1.000


### Zero-shot классификация

*Вес: 1 балл.*

**Постановка.** Подбор словесного шаблона влияет на прогноз без дополнительного обучения.

Выполните следующие действия:
1. Для каждого заранее заданного шаблона предскажите test-классы по сходству.
2. Зафиксируйте accuracy всех шаблонов без выбора лучшей модели по test.

**Результат.** zero_shot.json с фактической точностью шаблонов.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [50]:
# Реализуйте это задание по постановке выше.
@torch.no_grad()
def encode_images(model, loader, device=DEVICE):
    """Нормированные CLIP-эмбеддинги изображений и их метки."""
    feats, labels = [], []
    for imgs, lbls in loader:
        imgs = imgs.to(device)
        f = model.encode_image(imgs).float()
        f = F.normalize(f, dim=-1).cpu()
        feats.append(f)
        labels.append(lbls)
    return torch.cat(feats, dim=0), torch.cat(labels, dim=0)

# Эмбеддинги test считаем один раз, дальше переиспользуем везде
test_feats, test_labels = encode_images(model, loader)

@torch.inference_mode()
def zero_shot_accuracy(test_feats, test_labels, model, tokenizer, classes, template):
    texts = [template.format(c) for c in classes]
    tokens = tokenizer(texts).to(DEVICE)
    t_feats = F.normalize(model.encode_text(tokens).float(), dim=-1)
    logits = test_feats @ t_feats.T
    preds = logits.argmax(dim=1)
    return float((preds == test_labels).float().mean().item())

# Фиксируем accuracy ВСЕХ шаблонов, лучший по test не выбираем
zero_shot = {t: zero_shot_accuracy(test_feats, test_labels,
                                    model, tokenizer, classes, t)
             for t in templates}

with open(ARTIFACTS / "zero_shot.json", "w", encoding="utf-8") as f:
    json.dump(zero_shot, f, ensure_ascii=False, indent=2)

for t, a in sorted(zero_shot.items(), key=lambda x: -x[1]):
    print(f"{a:.4f}  {t}")
check(all(0 <= v <= 1 for v in zero_shot.values()), "Все accuracy в [0, 1]")

0.8739  a photo of a {} on a plain background.
0.8511  a photo of a {}.
0.8498  a blurry photo of a {}.
0.8373  a close-up photo of a {}.
0.8359  a photo of a small {}.
✔ Все accuracy в [0, 1]


### Векторный индекс изображений

*Вес: 1 балл.*

**Постановка.** Текстовый поиск опирается на сохранённые признаки реальных изображений.

Выполните следующие действия:
1. Извлеките нормированные эмбеддинги test и метки.
2. Вычислите top-10 для текстового запроса.

**Результат.** clip_index.pt и retrieval.json с индексами test.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [51]:
# Реализуйте это задание по постановке выше.
# Индекс — нормированные test-эмбеддинги + метки + список классов
torch.save({"feats": test_feats,
            "labels": test_labels,
            "classes": classes},
           ARTIFACTS / "clip_index.pt")

QUERY = "a photo of a cat."
with torch.inference_mode():
    q_tokens = tokenizer([QUERY]).to(DEVICE)
    q_feat = F.normalize(model.encode_text(q_tokens).float(), dim=-1).cpu()

sims = (test_feats @ q_feat.T).squeeze(1)
top10 = sims.topk(k=10).indices.tolist()

retrieval = {
    "query": QUERY,
    "top10_indices": top10,
    "top10_labels": test_labels[top10].tolist(),
    "top10_sims": sims[top10].tolist(),
}
with open(ARTIFACTS / "retrieval.json", "w", encoding="utf-8") as f:
    json.dump(retrieval, f, ensure_ascii=False, indent=2)

print(f"Запрос: {QUERY!r}")
print("Top-10 меток:", [classes[i] for i in retrieval["top10_labels"]])
check(len(top10) == 10, "Top-10 получен на реальных test-изображениях")

Запрос: 'a photo of a cat.'
Top-10 меток: ['dog', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat']
✔ Top-10 получен на реальных test-изображениях


### Релевантность поиска по классам

*Вес: 1 балл.*

**Постановка.** Precision@10 и recall@10 требуют фиксированного определения релевантности.

Выполните следующие действия:
1. Задайте релевантность как совпадение с классом запроса.
2. Вычислите метрики поиска для каждого класса.

**Результат.** retrieval_report.json с фактическими значениями.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [52]:
# Реализуйте это задание по постановке выше.
@torch.inference_mode()
def retrieval_metrics_per_class(test_feats, test_labels, model, tokenizer,
                                classes, template="a photo of a {}.", K=10):
    """P@K и R@K по каждому классу. Релевантность = совпадение класса."""
    results = {}
    for ci, cname in enumerate(classes):
        tokens = tokenizer([template.format(cname)]).to(DEVICE)
        q = F.normalize(model.encode_text(tokens).float(), dim=-1).cpu()
        sims = (test_feats @ q.T).squeeze(1)
        topK = sims.topk(k=K).indices
        topK_labels = test_labels[topK]
        hits = int((topK_labels == ci).sum().item())
        total_relevant = int((test_labels == ci).sum().item())
        results[cname] = {
            "precision@10": hits / K,
            "recall@10":    hits / total_relevant if total_relevant > 0 else 0.0,
            "hits":         hits,
            "total_relevant": total_relevant,
        }
    return results

retrieval_report = retrieval_metrics_per_class(
    test_feats, test_labels, model, tokenizer, classes
)

with open(ARTIFACTS / "retrieval_report.json", "w", encoding="utf-8") as f:
    json.dump(retrieval_report, f, ensure_ascii=False, indent=2)

mean_p = float(np.mean([v["precision@10"] for v in retrieval_report.values()]))
mean_r = float(np.mean([v["recall@10"]    for v in retrieval_report.values()]))
print(f"mean P@10 = {mean_p:.3f}, mean R@10 = {mean_r:.3f}")
check(all(0 <= v["precision@10"] <= 1 for v in retrieval_report.values()),
      "P@10 в [0, 1]")

mean P@10 = 0.960, mean R@10 = 0.010
✔ P@10 в [0, 1]


### Данные для обучаемой проекции

*Вес: 1 балл.*

**Постановка.** Одинаковые подписи в мини-партии не являются отрицательными парами. Проекция обучается по классовым прототипам.

Выполните следующие действия:
1. Создайте две линейные проекции и инициализируйте тождественно.
2. Подготовьте независимые train-пары и optimizer.

**Результат.** image_head, text_head и train_loader.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [53]:
# Реализуйте это задание по постановке выше.
# Кэшируем train-эмбеддинги один раз: CLIP forward на CPU дорогой
train_loader_raw = DataLoader(train_pairs, batch_size=32, shuffle=False, num_workers=0)
train_feats, train_labels_t = encode_images(model, train_loader_raw)

D = train_feats.shape[1]
print(f"train_feats={tuple(train_feats.shape)}, test_feats={tuple(test_feats.shape)}")

class IdentityLinear(torch.nn.Module):
    """Линейная проекция, инициализированная тождественно."""
    def __init__(self, dim):
        super().__init__()
        self.fc = torch.nn.Linear(dim, dim)
        with torch.no_grad():
            self.fc.weight.copy_(torch.eye(dim))
            self.fc.bias.zero_()
    def forward(self, x):
        return self.fc(x)

image_head = IdentityLinear(D).to(DEVICE)
text_head  = IdentityLinear(D).to(DEVICE)

# train_loader для обучения голов работает на тензорах, а не на CLIP forward
from torch.utils.data import TensorDataset
train_ds = TensorDataset(train_feats, train_labels_t)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)

check(D > 0 and len(train_ds) > 0,
      f"image_head/text_head инициализированы тождественно, D={D}")

train_feats=(50000, 512), test_feats=(10000, 512)
✔ image_head/text_head инициализированы тождественно, D=512


### Обучение проекций по меткам

*Вес: 1 балл.*

**Постановка.** Классовая cross-entropy не объявляет одинаковые captions взаимными негативами.

Выполните следующие действия:
1. Обучите проекции по logits изображения против всех десяти текстовых прототипов.
2. Сохраните обученные веса.

**Результат.** clip_projection.pt после оптимизации на train.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [65]:
# Реализуйте это задание по постановке выше.
text_protos_frozen = text_feats.to(DEVICE)   # [C, D]

# Обучаемая logit scale, как в оригинальном CLIP
logit_scale = torch.nn.Parameter(
    torch.tensor(np.log(1 / 0.07), dtype=torch.float32, device=DEVICE)
)

optimizer = torch.optim.AdamW(
    list(image_head.parameters()) + list(text_head.parameters()) + [logit_scale],
    lr=3e-3, weight_decay=1e-4,
)
criterion = torch.nn.CrossEntropyLoss()

EPOCHS = 3 if FAST_DEV_RUN else 25
history = []
for epoch in range(EPOCHS):
    image_head.train(); text_head.train()
    total, n = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        img_p = F.normalize(image_head(xb), dim=-1)
        txt_p = F.normalize(text_head(text_protos_frozen), dim=-1)
        scale = logit_scale.exp().clamp(max=100.0)
        logits = (img_p @ txt_p.T) * scale
        loss = criterion(logits, yb)
        loss.backward(); optimizer.step()
        total += loss.item() * xb.size(0); n += xb.size(0)
    history.append(total / max(1, n))
    print(f"Epoch {epoch+1}/{EPOCHS} | loss={history[-1]:.4f} | "
          f"scale={logit_scale.exp().item():.2f}")

torch.save({"image_head": image_head.state_dict(),
            "text_head":  text_head.state_dict(),
            "logit_scale": logit_scale.detach().cpu(),
            "dim":        D}, ARTIFACTS / "clip_projection.pt")
check(all(np.isfinite(h) for h in history),
      f"clip_projection.pt сохранён, финальный loss={history[-1]:.4f}")

Epoch 1/25 | loss=0.1668 | scale=10.02
Epoch 2/25 | loss=0.1460 | scale=10.28
Epoch 3/25 | loss=0.1391 | scale=10.79
Epoch 4/25 | loss=0.1340 | scale=11.40
Epoch 5/25 | loss=0.1311 | scale=11.64
Epoch 6/25 | loss=0.1282 | scale=12.10
Epoch 7/25 | loss=0.1254 | scale=12.30
Epoch 8/25 | loss=0.1225 | scale=13.14
Epoch 9/25 | loss=0.1206 | scale=13.53
Epoch 10/25 | loss=0.1202 | scale=14.13
Epoch 11/25 | loss=0.1177 | scale=14.29
Epoch 12/25 | loss=0.1170 | scale=14.90
Epoch 13/25 | loss=0.1150 | scale=15.24
Epoch 14/25 | loss=0.1148 | scale=15.22
Epoch 15/25 | loss=0.1133 | scale=15.62
Epoch 16/25 | loss=0.1115 | scale=15.97
Epoch 17/25 | loss=0.1113 | scale=16.92
Epoch 18/25 | loss=0.1096 | scale=16.97
Epoch 19/25 | loss=0.1093 | scale=16.98
Epoch 20/25 | loss=0.1086 | scale=17.28
Epoch 21/25 | loss=0.1084 | scale=17.74
Epoch 22/25 | loss=0.1069 | scale=18.48
Epoch 23/25 | loss=0.1069 | scale=18.36
Epoch 24/25 | loss=0.1059 | scale=19.13
Epoch 25/25 | loss=0.1047 | scale=19.53
✔ clip_pr

### Оценка адаптированного поиска

*Вес: 1 балл.*

**Постановка.** После обучения все варианты сравниваются на одинаковых test-изображениях.

Выполните следующие действия:
1. Обновите test-эмбеддинги двумя проекциями.
2. Сравните accuracy и среднюю precision@10 до и после обучения.

**Результат.** adaptation.json с измеренными baseline и adapted значениями.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [77]:
# Реализуйте это задание по постановке выше.
image_head.eval(); text_head.eval()

@torch.inference_mode()
def evaluate_retrieval(test_feats, test_labels, classes,
                       image_head, text_head, text_protos,
                       K=10):
    img_p = F.normalize(image_head(test_feats.to(DEVICE)), dim=-1).cpu()
    txt_p = F.normalize(text_head(text_protos.to(DEVICE)), dim=-1).cpu()

    logits = img_p @ txt_p.T
    acc = float((logits.argmax(dim=1) == test_labels).float().mean().item())

    mean_p10 = 0.0
    for ci in range(len(classes)):
        sims = img_p @ txt_p[ci]
        topK = sims.topk(k=K).indices
        hits = int((test_labels[topK] == ci).sum().item())
        mean_p10 += hits / K
    mean_p10 /= len(classes)
    return acc, mean_p10

# Baseline = тождественные головы
base_image_head = IdentityLinear(D).to(DEVICE)
base_text_head  = IdentityLinear(D).to(DEVICE)
base_acc, base_p10 = evaluate_retrieval(
    test_feats, test_labels, classes,
    base_image_head, base_text_head, text_feats
)
adapted_acc, adapted_p10 = evaluate_retrieval(
    test_feats, test_labels, classes,
    image_head, text_head, text_feats
)

adaptation = {
    "baseline_accuracy":  base_acc,
    "adapted_accuracy":   adapted_acc,
    "baseline_mean_p@10": base_p10,
    "adapted_mean_p@10":  adapted_p10,
    "delta_accuracy":     adapted_acc - base_acc,
    "delta_mean_p@10":    adapted_p10 - base_p10,
}
with open(ARTIFACTS / "adaptation.json", "w", encoding="utf-8") as f:
    json.dump(adaptation, f, ensure_ascii=False, indent=2)
print(adaptation)
check(all(np.isfinite(list(adaptation.values()))),
      "adaptation.json содержит измеренные baseline/adapted значения")

{'baseline_accuracy': 0.8540999889373779, 'adapted_accuracy': 0.942300021648407, 'baseline_mean_p@10': 0.9700000000000001, 'adapted_mean_p@10': 0.99, 'delta_accuracy': 0.08820003271102905, 'delta_mean_p@10': 0.019999999999999907}
✔ adaptation.json содержит измеренные baseline/adapted значения


### Лучшие и худшие результаты поиска

*Вес: 1 балл.*

**Постановка.** Конкретные ошибки поиска дополняют усреднённую precision@10.

Выполните следующие действия:
1. Выберите лучший и худший класс по precision@10.
2. Сохраните индексы и истинные классы найденных объектов.

**Результат.** search_examples.json с примерами реальных изображений.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [76]:
# Реализуйте это задание по постановке выше.
adapted_per_class = {}
with torch.inference_mode():
    img_p = F.normalize(image_head(test_feats.to(DEVICE)), dim=-1).cpu()
    txt_p = F.normalize(text_head(text_feats.to(DEVICE)), dim=-1).cpu()
    for ci, cname in enumerate(classes):
        sims = img_p @ txt_p[ci]
        top10 = sims.topk(k=1000).indices
        hits = int((test_labels[top10] == ci).sum().item())
        adapted_per_class[cname] = {
            "precision@10":  hits / 1000,
            "top10_indices": top10.tolist(),
            "top10_labels":  test_labels[top10].tolist(),
        }

best_c  = max(adapted_per_class, key=lambda c: adapted_per_class[c]["precision@10"])
worst_c = min(adapted_per_class, key=lambda c: adapted_per_class[c]["precision@10"])

search_examples = {
    "best_class":  {"name": best_c,  **adapted_per_class[best_c]},
    "worst_class": {"name": worst_c, **adapted_per_class[worst_c]},
}
with open(ARTIFACTS / "search_examples.json", "w", encoding="utf-8") as f:
    json.dump(search_examples, f, ensure_ascii=False, indent=2)

print(f"Best:  {best_c}  P@10={adapted_per_class[best_c]['precision@10']:.2f}")
print(f"Worst: {worst_c} P@10={adapted_per_class[worst_c]['precision@10']:.2f}")
check(best_c != worst_c, "Лучший и худший классы различаются")

Best:  ship  P@10=0.95
Worst: cat P@10=0.77
✔ Лучший и худший классы различаются


### Поклассовая динамика поиска

*Вес: 1 балл.*

**Постановка.** Средний показатель может скрывать ухудшение поиска для отдельных категорий.

Выполните следующие действия:
1. Повторно вычислите адаптированную precision@10 по каждому классу.
2. Сопоставьте её с исходной precision@10.

**Результат.** retrieval_deltas.json с наблюдаемыми поклассовыми изменениями.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [69]:
# Реализуйте это задание по постановке выше.
retrieval_deltas = {}
for cname in classes:
    before = retrieval_report[cname]["precision@10"]
    after  = adapted_per_class[cname]["precision@10"]
    retrieval_deltas[cname] = {
        "baseline_p@10": before,
        "adapted_p@10":  after,
        "delta":         after - before,
    }

with open(ARTIFACTS / "retrieval_deltas.json", "w", encoding="utf-8") as f:
    json.dump(retrieval_deltas, f, ensure_ascii=False, indent=2)

for c, v in sorted(retrieval_deltas.items(), key=lambda x: -x[1]["delta"]):
    print(f"{c:12s} base={v['baseline_p@10']:.2f} → "
          f"adapt={v['adapted_p@10']:.2f}  Δ={v['delta']:+.2f}")
check(len(retrieval_deltas) == len(classes),
      "Поклассовые дельты посчитаны для всех классов")

truck        base=0.80 → adapt=1.00  Δ=+0.20
dog          base=0.90 → adapt=1.00  Δ=+0.10
airplane     base=1.00 → adapt=1.00  Δ=+0.00
automobile   base=1.00 → adapt=1.00  Δ=+0.00
bird         base=1.00 → adapt=1.00  Δ=+0.00
cat          base=0.90 → adapt=0.90  Δ=+0.00
deer         base=1.00 → adapt=1.00  Δ=+0.00
frog         base=1.00 → adapt=1.00  Δ=+0.00
horse        base=1.00 → adapt=1.00  Δ=+0.00
ship         base=1.00 → adapt=1.00  Δ=+0.00
✔ Поклассовые дельты посчитаны для всех классов


### Творческий бонус: композиционный запрос

*Вес: 1 дополнительный балл.*

**Постановка.** Классовое имя не исчерпывает возможности текстово-визуального поиска.

Выполните следующие действия:
1. Сформулируйте собственный не-классовый запрос и получите top-10.
2. Сохраните коллаж найденных изображений для качественной оценки, не приписывая запросу искусственных меток.

**Результат.** bonus_retrieval.png и bonus_query.json с запросом и индексами.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [70]:
# Реализуйте это задание по постановке выше.
import matplotlib.pyplot as plt

BONUS_QUERY = "a green car on a road"
with torch.inference_mode():
    q_tokens = tokenizer([BONUS_QUERY]).to(DEVICE)
    q_feat = F.normalize(model.encode_text(q_tokens).float(), dim=-1).cpu()
    img_p = F.normalize(image_head(test_feats.to(DEVICE)), dim=-1).cpu()
    sims = img_p @ q_feat.squeeze(0)
    top10 = sims.topk(k=10).indices.tolist()

# Визуализация реальных test-изображений (обратно из нормализации CLIP)
mean = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(3, 1, 1)
std  = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(3, 1, 1)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, i in zip(axes.flat, top10):
    img_t, _ = full_test[test_indices[i]]
    vis = (img_t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(vis); ax.axis("off")
    ax.set_title(classes[int(test_labels[i])], fontsize=9)
plt.suptitle(f"Bonus query: {BONUS_QUERY!r}")
plt.tight_layout()
plt.savefig(ARTIFACTS / "bonus_retrieval.png")
plt.close()

with open(ARTIFACTS / "bonus_query.json", "w", encoding="utf-8") as f:
    json.dump({"query": BONUS_QUERY,
               "top10_indices": top10,
               "top10_labels": test_labels[top10].tolist()},
              f, ensure_ascii=False, indent=2)

print(f"✔ bonus_retrieval.png + bonus_query.json: {BONUS_QUERY!r}")

✔ bonus_retrieval.png + bonus_query.json: 'a green car on a road'


## Анализ результатов

Сопоставьте измеренные показатели на отложенных реальных данных. Укажите бюджет, ошибки модели и ограничения сокращённого запуска.


## Выводы

Сформулируйте предметный вывод по сохранённым артефактам.
